<a href="https://colab.research.google.com/github/kemikalce/Kemi-Project/blob/main/Copy_of_Trianed_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_excel('Cleaned_Student_Participation_Data.xlsx')
print(df.shape)
print(df.head())

(8558, 25)
  Learner SignUp DateTime                  Opportunity Id  \
0     2023-06-14 12:30:35  00000000-0GN2-A0AY-7XK8-C5FZPP   
1     2023-05-01 05:29:16  00000000-0GN2-A0AY-7XK8-C5FZPP   
2     2023-04-09 20:35:08  00000000-0GN2-A0AY-7XK8-C5FZPP   
3     2023-08-29 05:20:03  00000000-0GN2-A0AY-7XK8-C5FZPP   
4     2023-01-06 15:26:36  00000000-0GN2-A0AY-7XK8-C5FZPP   

                                    Opportunity Name Opportunity Category  \
0  Career Essentials: Getting Started With Your P...               Course   
1  Career Essentials: Getting Started With Your P...               Course   
2  Career Essentials: Getting Started With Your P...               Course   
3  Career Essentials: Getting Started With Your P...               Course   
4  Career Essentials: Getting Started With Your P...               Course   

  Opportunity End Date        First Name Date of Birth  Gender        Country  \
0  2024-06-29 18:52:39             Faria    2001-01-12  Female       Pakistan 

In [ ]:
print(df['Is_Successful'].value_counts())
print()
print(df['Is_Successful'].value_counts(normalize=True) * 100)

Is_Successful
0    5253
1    3305
Name: count, dtype: int64

Is_Successful
0    61.381164
1    38.618836
Name: proportion, dtype: float64


In [ ]:
# Select features for the model
features = ['Age', 'Is_International', 'Opp_Duration_Days',
            'Signup_to_Apply_Days', 'Apply Month',
            'Apply DayOfWeek', 'Status Code']

# Separate input features and target
X = df[features].dropna()
y = df.loc[X.index, 'Is_Successful']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (3267, 7)
Target shape: (3267,)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

Training rows: 2613
Testing rows: 654


In [ ]:
print(X.shape)

(3267, 7)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Training success rate:", y_train.mean().round(2))
print("Testing success rate:", y_test.mean().round(2))

Training rows: 2613
Testing rows: 654
Training success rate: 0.68
Testing success rate: 0.68


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print("Model trained successfully!")
print("Number of trees:", model.n_estimators)

Model trained successfully!
Number of trees: 100


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Make predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Model Accuracy:", round(accuracy * 100, 2), "%")
print()

# Detailed report
print("Classification Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 100.0 %

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       212
           1       1.00      1.00      1.00       442

    accuracy                           1.00       654
   macro avg       1.00      1.00      1.00       654
weighted avg       1.00      1.00      1.00       654



In [ ]:
# Remove Status Code - it leaks the answer
features = ['Age', 'Is_International', 'Opp_Duration_Days',
            'Signup_to_Apply_Days', 'Apply Month',
            'Apply DayOfWeek']

# Redo everything with fixed features
X = df[features].dropna()
y = df.loc[X.index, 'Is_Successful']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
print("Model Accuracy:", round(accuracy * 100, 2), "%")
print()
print(classification_report(y_test, y_pred))

Model Accuracy: 91.9 %

              precision    recall  f1-score   support

           0       0.90      0.85      0.87       212
           1       0.93      0.95      0.94       442

    accuracy                           0.92       654
   macro avg       0.91      0.90      0.91       654
weighted avg       0.92      0.92      0.92       654



In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Create predictions dataframe
results = df.loc[X_test.index].copy()
results['Predicted_Success'] = y_pred
results['Success_Probability'] = y_prob.round(3)

# Sort by probability - highest first
results = results.sort_values('Success_Probability', ascending=False)

# Save to Excel
results.to_excel('Predictions_Output.xlsx', index=False)
print("Saved successfully!")
print(results[['Opportunity Name', 'Success_Probability', 'Predicted_Success']].head(10))

Saved successfully!
                                       Opportunity Name  Success_Probability  \
8208  Jump Start: Developing Your Emotional Intellig...                  1.0   
7826                               Urbanrenew Challenge                  1.0   
8440                         Freelance Mastery Workshop                  1.0   
8083                         Xperience Design Hackathon                  1.0   
8207  Jump Start: Developing Your Emotional Intellig...                  1.0   
8128                         Xperience Design Hackathon                  1.0   
7777                               Urbanrenew Challenge                  1.0   
8376                         Freelance Mastery Workshop                  1.0   
8246  Jump Start: Developing Your Emotional Intellig...                  1.0   
8122                         Xperience Design Hackathon                  1.0   

      Predicted_Success  
8208                  1  
7826                  1  
8440                 

In [ ]:
import joblib

joblib.dump(model, 'trained_model.pkl')
print("Model saved as trained_model.pkl")

Model saved as trained_model.pkl


In [ ]:
from google.colab import files

files.download('Predictions_Output.xlsx')
files.download('trained_model.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>